# NovaUI 9 — Kaggle free preflight\n\nThis notebook performs non-destructive environment and repository checks only. It does not build or flash a ROM and does not download proprietary firmware.

In [ ]:
import os, platform, shutil, subprocess, pathlib, json\nprint('OS:', platform.platform())\nprint('Architecture:', platform.machine())\nprint('CPU cores:', os.cpu_count())\ntry:\n    import psutil\n    print('RAM total (GiB):', round(psutil.virtual_memory().total / 2**30, 2))\nexcept Exception as e:\n    print('RAM probe unavailable:', e)\nprint('Working disk free (GiB):', round(shutil.disk_usage('/kaggle/working').free / 2**30, 2))\nprint('No device or flash operations are performed.')

In [ ]:
REPO = 'https://github.com/Yasith200109/NovaUI-9-N976N.git'\nBRANCH = 'porting/oneui9-d2xks'\nDEST = pathlib.Path('/kaggle/working/NovaUI-9-N976N')\nif not DEST.exists():\n    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(DEST)], check=True)\nelse:\n    subprocess.run(['git', '-C', str(DEST), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)\n    subprocess.run(['git', '-C', str(DEST), 'checkout', 'FETCH_HEAD'], check=True)\nprint('Checked out:', subprocess.check_output(['git','-C',str(DEST),'rev-parse','--short','HEAD'], text=True).strip())

In [ ]:
manifest = DEST / 'device/d2xks/device.yaml'\nreadme = DEST / 'README.md'\nassert manifest.is_file() and manifest.stat().st_size > 0, 'Missing device manifest'\nassert readme.is_file() and readme.stat().st_size > 0, 'Missing README'\ntext = manifest.read_text(encoding='utf-8')\nassert 'SM-N976N' in text, 'Manifest does not identify SM-N976N'\nassert 'flashable_artifacts_available: false' in text, 'Unexpected flashable-artifact status'\nprint('PASS: repository and device metadata checks')

In [ ]:
script = DEST / 'scripts/check-source.sh'\nif script.exists():\n    result = subprocess.run(['bash','-n',str(script)], capture_output=True, text=True)\n    print(result.stdout or 'Shell syntax check completed')\n    if result.returncode:\n        raise RuntimeError(result.stderr)\n    print('PASS: shell script syntax')\nelse:\n    print('No source-check script found; skipped')

## Safety reminder\nDo not attach public outputs containing Samsung firmware, vendor blobs, modem files, signing keys, personal identifiers, or private logs. This is a preflight environment only, not a full ROM build.